# Whale and Boat Detection: Evaluation

Compares the CNN baseline (Faster R-CNN) with the DETR model on the **test split, used once**.
Every cutoff is chosen on the validation split first. Every number states its metric, split and
count.

1. Setup and runs
2. Predictions, plus a non-learned baseline (bright/dark blob detector)
3. Operating points: three cutoff rules, chosen on validation
4. Main results with group-bootstrap confidence intervals
5. Precision-recall curves
6. Counting whales and boats
7. Recall by annotator certainty, including the "possibly a whale" chips
8. Results by imaging conditions (haze, water roughness, sun elevation, date)
9. Failure examples
10. What DETR attends to
11. Ablation: weather augmentation on vs off
12. Tuning and learning curves
13. Limitations to discuss

**Terms.** A *chip* is one 500 x 500 px tile (250 x 250 m, 0.0625 km²). A *scene* is a whole
SkySat image, a median of ~65 km² (~1,000 chips). False alarms are reported per km² and per
median scene: the number of boxes a researcher would check by eye that hold no whale or boat.

**Scoring.** A detection finds a whale or boat if its center is within the run's match radius
(10 m by default) of an annotated point of the same class. Each point can be found once; extra
detections on it are false alarms. "Possibly a whale" points (certainty 0) count neither way.

## 1. Setup and runs

In [ ]:
import os
import sys

os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rasterio timm "transformers>=5.20" optuna

In [ ]:
import csv
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(ROOT / 'src'))
from whaleboat.data import overlaps_same_day, read_chip  # noqa: E402
from whaleboat.detection import (  # noqa: E402
    CHIP_KM2,
    DET_CLASSES,
    PIXEL_M,
    ChipCache,
    agnostic_chips,
    anomaly_detections,
    counts_by,
    evaluate_detections,
    group_bootstrap,
    load_detection_chips,
    load_uncertain_whales,
    make_det_loader,
    merge_classes,
    pr_curve,
    summarize,
    threshold_fbeta,
    threshold_for_count,
    threshold_for_fa_budget,
)
from whaleboat.detectors import build_detector, predict_loader  # noqa: E402

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device {DEVICE}')

`RUNS` are the models compared in the main table (run folders written by the training
notebooks). `ABLATION` is the weather-augmentation pair; leave a run out if it hasn't been
trained yet.

In [ ]:
RUNS = {'Faster R-CNN': 'frcnn_main', 'Conditional DETR': 'detr_main'}
ABLATION = {'DETR, weather aug': 'detr_main', 'DETR, no weather aug': 'detr_noweather'}
DETR_RUN = 'Conditional DETR'  # whose attention to show
BOOTSTRAP_N = 500
# Operating points, each chosen on validation (section 3):
FBETA = 2.0  # 'find': maximize F2 (recall weighted 4x precision)
REVIEW_BUDGET_PER_KM2 = 16.0  # 'review': false alarms a reviewer accepts per km2 (16 ~ 1 per chip)
MAIN_RULE = 'find'  # the cutoff used for breakdowns, failure examples and the ablation
SCENE_KM2 = 64.5  # median SkySat scene area, to express false alarms per scene
BATCH_SIZE = 8
USE_AMP = True
NUM_WORKERS = 2

DATA_DIR = ROOT / 'data'
ORIG_DIR = DATA_DIR / 'whale' / 'data'
SKYSAT_DIR = DATA_DIR / 'skysat_chips_500'
CKPT_DIR = ROOT / 'checkpoints'
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    CKPT_DIR = Path('/content/drive/MyDrive/whaleboat/checkpoints')

all_runs = {**RUNS, **ABLATION}
runs = {}
for label, name in all_runs.items():
    path = CKPT_DIR / name / 'run.json'
    if path.exists():
        runs[label] = json.loads(path.read_text())
    else:
        print(f'skipping {label}: {path} not found')
RUNS = {k: v for k, v in RUNS.items() if k in runs}
ABLATION = {k: v for k, v in ABLATION.items() if k in runs}

ref = runs[next(iter(RUNS))]
for label, r in runs.items():
    assert r['split']['test'] == ref['split']['test'], f'{label} has a different test split'
    assert r['match_radius_m'] == ref['match_radius_m'], f'{label} uses another match radius'
RADIUS_PX = ref['match_radius_m'] / PIXEL_M

pd.DataFrame(
    {
        label: {
            'model': r['model'],
            'weather aug': r['weather_aug'],
            'datasets': r['datasets'],
            'split': r['split_from'] or r['split_mode'],
            **{k: v for k, v in r['params'].items()},
        }
        for label, r in runs.items()
    }
)

In [ ]:
chips = load_detection_chips(ref['datasets'], ORIG_DIR, SKYSAT_DIR)
by_name = {c['name']: c for c in chips}
parts = {k: [by_name[n] for n in v] for k, v in ref['split'].items()}
test, val = parts['test'], parts['val']

# "Possibly a whale" chips, never trained on. Keep only those not overlapping a training or
# validation chip from the same day, so the same animal isn't scored after being trained on.
uncertain = [
    c
    for c in load_uncertain_whales(ORIG_DIR, SKYSAT_DIR)
    if not overlaps_same_day(c, parts['train'] + parts['val'])
]

conditions = {}
with open(SKYSAT_DIR / 'main' / 'manifest_conditions.csv', newline='') as f:
    for row in csv.DictReader(f):
        conditions[f'skysat/{row["chip_id"]}'] = row

n_pts = lambda cs, k: sum(p['cls'] == k and not p['ignore'] for c in cs for p in c['points'])  # noqa: E731
print(
    f'test: {len(test)} chips, {n_pts(test, "whale")} whales, {n_pts(test, "boat")} boats, '
    f'{len({c["group"] for c in test})} groups'
)
print(f'val:  {len(val)} chips, {n_pts(val, "whale")} whales, {n_pts(val, "boat")} boats')
print(f'possibly-whale chips kept: {len(uncertain)} ({n_pts(uncertain, "whale")} whales)')

## 2. Predictions

Each model sees chips with its own run's band stretch and input scale, and keeps its own
validation cutoffs. The baseline is a non-learned blob detector (local brightness anomaly in
robust z units): it can't tell whales from boats, so it is compared class-agnostically, with
its cutoff also set on validation.

In [ ]:
caches = {}


def cache_for(run):
    key = (tuple(run['band_lo']), tuple(run['band_hi']))
    if key not in caches:
        caches[key] = ChipCache(test + val + uncertain, run['band_lo'], run['band_hi'])
    return caches[key]


def load_detector(run):
    ckpt = torch.load(
        CKPT_DIR / run['run_name'] / 'detector.pt', map_location='cpu', weights_only=False
    )
    size = round(500 * run['params']['scale'])
    det = build_detector(
        run['model'],
        size,
        run['params'],
        pretrained=False,
        detr_checkpoint=run.get('detr_checkpoint'),
    )
    det.load_state_dict(ckpt['model_state_dict'])
    return det.to(DEVICE).eval()


def predict(run, det, cs):
    loader = make_det_loader(
        cs, cache_for(run), BATCH_SIZE, run['params']['scale'], workers=NUM_WORKERS
    )
    return predict_loader(det, loader, run['params']['scale'], DEVICE, amp=USE_AMP)


preds = {}
for label, run in runs.items():
    det = load_detector(run)
    preds[label] = {
        k: predict(run, det, cs)
        for k, cs in [('test', test), ('val', val), ('uncertain', uncertain)]
    }
    del det
    torch.cuda.empty_cache()
    print(f'{label}: predicted {sum(len(v) for v in preds[label].values())} chips')

baseline = {
    k: {c['name']: anomaly_detections(read_chip(c['path'])) for c in cs}
    for k, cs in [('test', test), ('val', val)]
}

## 3. Operating points (chosen on validation)

A detector gives every candidate a score; the cutoff decides which ones are kept. Keeping
everything finds the most whales but buries them in false alarms, so each model gets three
cutoffs, all chosen on the **validation** split and then applied unchanged to test:

- **find**: maximizes F2, which weighs recall 4x precision. The standard choice when a missed
  whale costs more than a false alarm; it never falls back to keeping everything.
- **review**: the lowest cutoff whose false alarms stay within `REVIEW_BUDGET_PER_KM2`. The most
  whales a researcher sees for a fixed amount of checking; set the budget with the researchers.
- **count**: the cutoff where the number of detections kept matches the number of objects, so
  misses and false alarms cancel in the total. Used for counting (section 6), not for finding.

In [ ]:
RULES = {
    'find': lambda r: threshold_fbeta(r, FBETA),
    'review': lambda r: threshold_for_fa_budget(r, REVIEW_BUDGET_PER_KM2),
    'count': threshold_for_count,
}
cutoffs, rows = {}, []
for label in runs:
    vres = evaluate_detections(preds[label]['val'], val, RADIUS_PX)
    cutoffs[label] = {rule: {k: f(vres[k]) for k in DET_CLASSES} for rule, f in RULES.items()}
    for rule in RULES:
        m = summarize(vres, cutoffs[label][rule])
        for k in DET_CLASSES:
            rows.append(
                {
                    'model': label,
                    'rule': rule,
                    'class': k,
                    'cutoff': round(cutoffs[label][rule][k], 3),
                    'val recall': m[f'{k}_recall'],
                    'val precision': m[f'{k}_precision'],
                    'val false alarms / km2': m[f'{k}_fa_per_km2'],
                }
            )
print(f'Validation: {len(val)} chips. Cutoffs below are applied unchanged to test.')
pd.DataFrame(rows).round(3)

## 4. Main results (test split)

Per class: **AP** (area under the precision-recall curve, every cutoff), then at the **find** and
**review** cutoffs from section 3: **recall** (share of test whales/boats found), **precision**,
and **false alarms** per km² and per median scene. Brackets are 95% intervals from resampling
whole test groups (same-day overlapping chips), since chips within a group are not independent.

In [ ]:
def fmt(v, ci=None, digits=2):
    if v is None or v != v:
        return 'n/a'
    s = f'{v:.{digits}f}'
    return s if ci is None else f'{s} [{ci[0]:.{digits}f}, {ci[1]:.{digits}f}]'


results, rows = {}, []
for label in RUNS:
    res = evaluate_detections(preds[label]['test'], test, RADIUS_PX)
    results[label] = res
    for rule in ['find', 'review']:
        thr = cutoffs[label][rule]
        m = summarize(res, thr)
        ci = group_bootstrap(preds[label]['test'], test, RADIUS_PX, thr, n=BOOTSTRAP_N)
        for k in DET_CLASSES:
            fa = m[f'{k}_fa_per_km2']
            rows.append(
                {
                    'model': label,
                    'class': k,
                    'cutoff': f'{rule} ({thr[k]:.3f})',
                    'n objects': m[f'{k}_n'],
                    'AP': fmt(m[f'{k}_ap'], ci[f'{k}_ap']),
                    'recall': fmt(m[f'{k}_recall'], ci[f'{k}_recall']),
                    'precision': fmt(m[f'{k}_precision'], ci[f'{k}_precision']),
                    'false alarms / km2': fmt(fa, ci[f'{k}_fa_per_km2'], 1),
                    'false alarms / scene': round(fa * SCENE_KM2),
                }
            )
print(
    f'Test split: {len(test)} chips ({len(test) * CHIP_KM2:.1f} km2), '
    f'{len({c["group"] for c in test})} groups, {BOOTSTRAP_N} bootstrap resamples'
)
main_table = pd.DataFrame(rows)
main_table

**Any object (class-agnostic), with the blob baseline.** Whale and boat detections are pooled
(a whale and a boat detection on the same spot count once), and every annotated object counts
regardless of class. This asks whether the learned models beat simple image statistics at
*finding* objects at all. Each method uses its own **find** (F2) cutoff from validation.

In [ ]:
test_any, val_any = agnostic_chips(test), agnostic_chips(val)
sources = {
    label: {k: merge_classes(preds[label][k], RADIUS_PX) for k in ['test', 'val']} for label in RUNS
}
sources['Blob baseline (no training)'] = baseline
rows = []
for label, p in sources.items():
    vres = evaluate_detections(p['val'], val_any, RADIUS_PX, ['object'])
    thr = {'object': threshold_fbeta(vres['object'], FBETA)}
    m = summarize(evaluate_detections(p['test'], test_any, RADIUS_PX, ['object']), thr)
    ci = group_bootstrap(p['test'], test_any, RADIUS_PX, thr, n=BOOTSTRAP_N, classes=['object'])
    rows.append(
        {
            'method': label,
            'n objects': m['object_n'],
            'AP': fmt(m['object_ap'], ci['object_ap']),
            'recall': fmt(m['object_recall'], ci['object_recall']),
            'precision': fmt(m['object_precision'], ci['object_precision']),
            'false alarms / km2': fmt(m['object_fa_per_km2'], ci['object_fa_per_km2'], 1),
        }
    )
pd.DataFrame(rows)

## 5. Precision-recall curves (test split)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, k in zip(axes, DET_CLASSES):
    for label, res in results.items():
        prec, rec, _ = pr_curve(res[k])
        (line,) = ax.plot(rec, prec, label=f'{label} (AP {summarize({k: res[k]})[f"{k}_ap"]:.2f})')
        for rule, marker in [('find', 'o'), ('review', '^')]:
            m = summarize({k: res[k]}, cutoffs[label][rule])
            ax.scatter(
                [m[f'{k}_recall']],
                [m[f'{k}_precision']],
                color=line.get_color(),
                marker=marker,
                zorder=3,
            )
    ax.set(
        xlabel='recall',
        ylabel='precision',
        xlim=(0, 1),
        ylim=(0, 1.02),
        title=f'{k} (test, n={res[k]["n_points"]}); o = find, ^ = review cutoff',
    )
    ax.legend(loc='lower left')
plt.show()

## 6. Counting whales and boats (test split)

"How many whales are in this image?" uses the **count** cutoff: misses and false alarms cancel,
so totals can be right even when individual boxes are wrong. Counts are per capture (one
satellite pass), over the test chips from that capture only, not the whole scene. Possible
whales are left out of both the true and the predicted counts.

- **count cutoff**: number of detections kept at the count cutoff.
- **score sum**: the sum of all detection scores, no cutoff. It is a fair estimate only if the
  scores behave like probabilities; a large gap from the true total says they don't.

A right total can hide errors that cancel across captures, so the per-capture error and the
scatter matter more than the total.

In [ ]:
def capture_of(name):
    return conditions[name]['capture'] if name in conditions else by_name[name]['group']


rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, k in zip(axes, DET_CLASSES):
    top = 1
    for label in RUNS:
        c = counts_by(results[label][k], cutoffs[label]['count'][k], test, capture_of)
        true = np.array([v[0] for v in c.values()])
        pred = np.array([v[1] for v in c.values()])
        ssum = np.array([v[2] for v in c.values()])
        rows.append(
            {
                'model': label,
                'class': k,
                'captures': len(c),
                'true total': int(true.sum()),
                'predicted total': int(pred.sum()),
                'score-sum total': round(float(ssum.sum()), 1),
                'error per capture (mean abs)': float(np.abs(pred - true).mean()),
                'score-sum error per capture': float(np.abs(ssum - true).mean()),
                'captures counted exactly': float((pred == true).mean()),
            }
        )
        ax.scatter(true, pred, alpha=0.7, label=label)
        top = max(top, true.max(), pred.max())
    ax.plot([0, top], [0, top], color='gray', lw=1, ls=':')
    ax.set(xlabel=f'true {k}s per capture', ylabel=f'predicted {k}s', title=f'{k} counts (test)')
    ax.legend(fontsize=8)
plt.show()
pd.DataFrame(rows).round(2)

## 7. Recall by annotator certainty

Certainty 2 = "clearly a whale", 1 = "likely", 0 = "possibly". Certainty 0 whales come from the
separate `uncertain_whales` chips (only those not overlapping a training or validation chip).
They were never training targets. Recall at the **find** cutoff; precision isn't meaningful
here because those chips weren't fully checked for other objects.

In [ ]:
rows = []
for label in RUNS:
    thr = cutoffs[label][MAIN_RULE]['whale']
    res = results[label]['whale']
    for cert in [2, 1]:
        found = [
            s >= thr for s, p in zip(res['point_score'], res['points']) if p['certainty'] == cert
        ]
        rows.append(
            {
                'model': label,
                'certainty': cert,
                'whales': len(found),
                'recall': np.mean(found) if found else np.nan,
            }
        )
    ures = evaluate_detections(preds[label]['uncertain'], uncertain, RADIUS_PX)['whale']
    found = ures['point_score'] >= thr
    rows.append(
        {
            'model': label,
            'certainty': 0,
            'whales': len(found),
            'recall': found.mean() if len(found) else np.nan,
        }
    )
pd.DataFrame(rows).pivot(index='certainty', columns='model', values=['recall', 'whales']).round(2)

## 8. Results by imaging conditions (test split)

Per subgroup: whale and boat recall at the **find** cutoff, and false alarms per km².
Subgroups are small, so read these as patterns to look at, not firm differences. Haze and
roughness are only known for SkySat chips; the original 144 chips are listed separately.

- **Haze:** share of the chip flagged heavy haze in the UDM2 mask (scene-level light-haze % too).
- **Water roughness:** NIR standard deviation (glint and whitecaps raise it), test terciles.
- **Sun elevation:** low sun lengthens glint and shadows, test terciles.

In [ ]:
def cond(name, key):
    v = conditions.get(name, {}).get(key, '')
    try:
        return float(v)
    except ValueError:
        return np.nan


def terciles(key):
    vals = np.array([cond(c['name'], key) for c in test])
    return np.nanpercentile(vals, [33.3, 66.7]) if np.isfinite(vals).any() else [np.nan] * 2


rough_cut, sun_cut = terciles('nir_std'), terciles('sun_elevation')


def bucket(v, cuts, names):
    if v != v:
        return 'original chips'
    return names[0] if v < cuts[0] else names[1] if v < cuts[1] else names[2]


GROUPINGS = {
    'haze': lambda n: (
        'original chips'
        if n not in conditions
        else (
            'heavy haze'
            if cond(n, 'heavy_haze_frac') > 0.5
            else 'light haze'
            if cond(n, 'light_haze_percent') > 20
            else 'clear'
        )
    ),
    'water roughness': lambda n: bucket(cond(n, 'nir_std'), rough_cut, ['calm', 'medium', 'rough']),
    'sun elevation': lambda n: bucket(
        cond(n, 'sun_elevation'), sun_cut, ['low sun', 'mid sun', 'high sun']
    ),
    'date': lambda n: by_name[n]['date'],
}


def breakdown(label, group_of):
    res, thr = results[label], cutoffs[label][MAIN_RULE]
    out = {}
    for c in test:
        g = out.setdefault(group_of(c['name']), {'chips': 0})
        g['chips'] += 1
    for k in DET_CLASSES:
        for s, p in zip(res[k]['point_score'], res[k]['points']):
            g = out[group_of(p['chip'])]
            g[f'{k}s'] = g.get(f'{k}s', 0) + 1
            g[f'{k} found'] = g.get(f'{k} found', 0) + (s >= thr[k])
        for name, s, hit in zip(res[k]['det_chip'], res[k]['scores'], res[k]['hit']):
            if s >= thr[k] and not hit:
                g = out[group_of(name)]
                g['false alarms'] = g.get('false alarms', 0) + 1
    rows = []
    for g, d in sorted(out.items()):
        rows.append(
            {
                'subgroup': g,
                'chips': d['chips'],
                'whales': d.get('whales', 0),
                'whale recall': d.get('whale found', 0) / d['whales']
                if d.get('whales')
                else np.nan,
                'boats': d.get('boats', 0),
                'boat recall': d.get('boat found', 0) / d['boats'] if d.get('boats') else np.nan,
                'false alarms / km2': d.get('false alarms', 0) / (d['chips'] * CHIP_KM2),
            }
        )
    return pd.DataFrame(rows).set_index('subgroup')


for grouping, fn in GROUPINGS.items():
    print(f'--- by {grouping} (test) ---')
    display(pd.concat({label: breakdown(label, fn) for label in RUNS}, axis=1).round(2))

## 9. Failure examples (test split)

Missed whales (no detection at the **find** cutoff within the match radius) and the highest-scoring false
alarms, each as a 100 x 100 px (50 m) crop. Red circle = annotated whale, cyan = detection.

In [ ]:
def crop(name, x, y, run, half=50):
    img = cache_for(run)[name].transpose(1, 2, 0)
    x0, y0 = int(np.clip(x - half, 0, 500 - 2 * half)), int(np.clip(y - half, 0, 500 - 2 * half))
    return img[y0 : y0 + 2 * half, x0 : x0 + 2 * half], x - x0, y - y0


def gallery(items, run, title, color, n=8):
    items = items[:n]
    if not items:
        print(f'{title}: none')
        return
    fig, axes = plt.subplots(1, len(items), figsize=(2.2 * len(items), 2.6))
    for ax, (name, x, y, text) in zip(np.atleast_1d(axes), items):
        im, cx, cy = crop(name, x, y, run)
        ax.imshow(im)
        ax.add_patch(plt.Circle((cx, cy), RADIUS_PX, fill=False, color=color, lw=1.2))
        ax.set_title(text, fontsize=7)
        ax.axis('off')
    fig.suptitle(title)
    plt.show()


for label in RUNS:
    run, res = runs[label], results[label]['whale']
    thr = cutoffs[label][MAIN_RULE]['whale']
    missed = [
        (p['chip'], p['x'], p['y'], f'cert {p["certainty"]}, best {s:.2f}')
        for s, p in sorted(zip(res['point_score'], res['points']), key=lambda t: t[0])
        if s < thr
    ]
    random.Random(0).shuffle(missed)
    gallery(missed, run, f'{label}: missed whales ({len(missed)} of {res["n_points"]})', 'red')
    for k in DET_CLASSES:
        r = results[label][k]
        fa = [
            (n, xy[0], xy[1], f'{k} {s:.2f}')
            for n, xy, s, h in zip(r['det_chip'], r['det_xy'], r['scores'], r['hit'])
            if not h and s >= cutoffs[label][MAIN_RULE][k]
        ]
        fa.sort(key=lambda t: -float(t[3].split()[-1]))
        gallery(fa, run, f'{label}: top false alarms as {k}', 'cyan')

## 10. What DETR attends to

Each DETR prediction comes from one object query. The maps show that query's cross-attention
in the last decoder layer (mean over the 8 heads) on the backbone's feature grid (one cell per
32 input pixels). Left: chips where DETR finds a whale; right: hazy or rough-water chips. Where
the attention spreads over glint or haze instead of the animal, the detection is fragile.
(Shown for Conditional DETR; Deformable DETR uses sampled points instead of a full map.)

In [ ]:
def attention_maps(run, det, names):
    det.model.set_attn_implementation('eager')
    scale = run['params']['scale']
    loader = make_det_loader([by_name[n] for n in names], cache_for(run), 1, scale)
    out = []
    for images, targets in loader:
        with torch.no_grad():
            raw = det.raw(images.to(DEVICE), output_attentions=True)
        if not raw.cross_attentions:
            return []
        d = det.decode(raw)[0]
        q = int(d['query'][0])
        attn = raw.cross_attentions[-1][0, :, q].mean(0)
        grid = int(round(attn.numel() ** 0.5))
        attn = attn.reshape(1, 1, grid, grid).float()
        attn = F.interpolate(attn, size=images.shape[-2:], mode='bilinear')[0, 0].cpu().numpy()
        box = (d['boxes'][0] / scale).cpu().numpy()
        out.append(
            (
                targets[0]['name'],
                images[0].permute(1, 2, 0).numpy(),
                attn,
                float(d['scores'][0]),
                DET_CLASSES[int(d['labels'][0])],
                box,
                scale,
            )
        )
    return out


if DETR_RUN in runs and runs[DETR_RUN]['model'] == 'detr':
    run = runs[DETR_RUN]
    res = results[DETR_RUN]['whale']
    found = [
        p['chip']
        for s, p in zip(res['point_score'], res['points'])
        if s >= cutoffs[DETR_RUN][MAIN_RULE]['whale']
    ]
    hard = [
        c['name']
        for c in test
        if c['name'] in conditions
        and c['points']
        and (cond(c['name'], 'heavy_haze_frac') > 0.5 or cond(c['name'], 'nir_std') > rough_cut[1])
    ]
    found = list(dict.fromkeys(found))[:3]
    pick = [(n, 'whale found') for n in found] + [
        (n, 'hazy / rough water') for n in hard if n not in found
    ][:3]
    det = load_detector(run)
    maps = attention_maps(run, det, [n for n, _ in pick])
    del det
    if maps:
        fig, axes = plt.subplots(2, len(maps), figsize=(3.2 * len(maps), 6.6))
        for i, (name, img, attn, s, k, box, scale) in enumerate(maps):
            axes[0, i].imshow(img)
            x0, y0, x1, y1 = box * scale
            axes[0, i].add_patch(
                plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='cyan')
            )
            for p in by_name[name]['points']:
                axes[0, i].add_patch(
                    plt.Circle(
                        (p['x'] * scale, p['y'] * scale), 12 * scale, fill=False, color='red'
                    )
                )
            axes[0, i].set_title(f'top: {k} {s:.2f}', fontsize=8)
            axes[1, i].imshow(img)
            axes[1, i].imshow(attn, cmap='magma', alpha=0.6)
            axes[1, i].set_title(pick[i][1], fontsize=8)
            for ax in axes[:, i]:
                ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('This DETR variant does not return cross-attention maps.')

## 11. Ablation: weather augmentation (test split)

Same split, same tuned settings, same cutoff rule; only weather augmentation differs. The
question is whether it helps on hazy, glinty or rough-water chips without hurting clear ones.
The difference column is the paired group-bootstrap 95% interval of (with - without): if it
includes 0, the test set can't tell the two apart.

In [ ]:
def paired_diff(label_a, label_b, keys, n=BOOTSTRAP_N, seed=0):
    rng = np.random.default_rng(seed)
    groups = {}
    for c in test:
        groups.setdefault(c['group'], []).append(c)
    names = list(groups)
    diffs = {k: [] for k in keys}
    for _ in range(n):
        sample, pa, pb = [], {}, {}
        for b, g in enumerate(rng.choice(len(names), len(names))):
            for c in groups[names[g]]:
                key = f'{c["name"]}#{b}'
                sample.append({**c, 'name': key})
                pa[key] = preds[label_a]['test'].get(c['name'], [])
                pb[key] = preds[label_b]['test'].get(c['name'], [])
        ma = summarize(evaluate_detections(pa, sample, RADIUS_PX), cutoffs[label_a][MAIN_RULE])
        mb = summarize(evaluate_detections(pb, sample, RADIUS_PX), cutoffs[label_b][MAIN_RULE])
        for k in keys:
            diffs[k].append(ma[k] - mb[k])
    return {k: tuple(np.nanpercentile(v, [2.5, 97.5])) for k, v in diffs.items()}


if len(ABLATION) == 2:
    (la, _), (lb, _) = ABLATION.items()
    keys = [
        'whale_ap',
        'boat_ap',
        'whale_recall',
        'boat_recall',
        'whale_fa_per_km2',
        'boat_fa_per_km2',
    ]
    ma = summarize(evaluate_detections(preds[la]['test'], test, RADIUS_PX), cutoffs[la][MAIN_RULE])
    mb = summarize(evaluate_detections(preds[lb]['test'], test, RADIUS_PX), cutoffs[lb][MAIN_RULE])
    ci = paired_diff(la, lb, keys)
    display(
        pd.DataFrame(
            [
                {
                    'metric': k,
                    la: round(ma[k], 3),
                    lb: round(mb[k], 3),
                    'difference [95% CI]': fmt(ma[k] - mb[k], ci[k], 3),
                }
                for k in keys
            ]
        )
    )
    for label in ABLATION:
        results.setdefault(label, evaluate_detections(preds[label]['test'], test, RADIUS_PX))
    for grouping in ['haze', 'water roughness']:
        print(f'--- by {grouping} (test) ---')
        display(
            pd.concat(
                {label: breakdown(label, GROUPINGS[grouping]) for label in ABLATION}, axis=1
            ).round(2)
        )
else:
    print('Train both ablation runs to fill this section.')

## 12. Tuning and learning curves

The settings each search chose, how many trials ran, and validation AP during the final training
run (for the report's hyperparameter table and to spot under/overfitting).

In [ ]:
rows = {}
for label, run in runs.items():
    trials_csv = CKPT_DIR / run['run_name'] / 'search_trials.csv'
    t = pd.read_csv(trials_csv) if trials_csv.exists() else None
    rows[label] = {
        **run['params'],
        'trials (finished / stopped early)': 'reused'
        if t is None
        else f'{(t.state == "COMPLETE").sum()} / {(t.state == "PRUNED").sum()}',
        f'val {run["fixed"]["selection_metric"]}': max(h['val_score'] for h in run['history']),
        'epochs trained': len(run['history']),
    }
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for label, run in runs.items():
    h = pd.DataFrame(run['history'])
    axes[0].plot(h.epoch, h.loss, label=label)
    axes[1].plot(h.epoch, h.val_whale_ap, label=label)
axes[0].set(xlabel='epoch', ylabel='training loss', title='Training loss')
axes[1].set(xlabel='epoch', ylabel='validation whale AP', title='Validation whale AP', ylim=(0, 1))
axes[1].legend(fontsize=8)
plt.show()

## 13. Limitations to discuss

- **Labels:** one annotator pass per scene; about half of all whale points are "possibly". A
  "false alarm" may be a whale the annotator missed, so precision is a lower bound.
- **Points, not boxes:** scoring depends on the match radius; report it and, if time allows,
  show results at 5 / 10 / 15 m.
- **Test size:** a few dozen test whales from a handful of days, so intervals are wide and
  subgroup tables are indicative only.
- **Cutoffs:** chosen on ~60 validation whales, so they are noisy; F2's optimum is often flat
  over a wide range of cutoffs. The review budget should come from the researchers.
- **Counting:** totals can be right while individual captures are wrong; report per-capture error.
- **One region:** all SkySat scenes are off West Maui; other waters, sensors or seasons may
  behave differently.
- **Chips, not scenes:** chips were cut around annotations and fully checked water. A full-scene
  run (tiled inference over every Done grid cell) is the real use case and will show more false
  alarms per km² from land edges, wakes and clouds. That is the planned demo.